[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/docencia/nb06-modelos-preentrenados-2026/Machine_Learning_2026/03_Notebooks/NB06_Transfer_Learning_Fine_Tuning_Evaluacion_CNN.ipynb)
# NB06 — Evaluación de modelos preentrenados con imágenes

## Propósito
Experimentar con **MobileNetV2** (entrenada en ImageNet) **sin entrenar ni ajustar pesos**. Primero analizaremos un conjunto común de **20 imágenes CIFAR-10 de GitHub** (dos por categoría); después **deberás** repetir el experimento con tu propio dataset del NB05 y probar una imagen individual subida a Colab.

**Objetivos:** observar predicciones y probabilidades, distinguir aciertos/errores con criterio semántico y reconocer limitaciones de resolución y de clases. **No se realiza Fine-Tuning.**

**Importante:** CIFAR-10 tiene diez categorías; ImageNet tiene mil clases que no corresponden necesariamente a las mismas etiquetas. Una coincidencia textual exacta no es una medida adecuada de acierto.

## 1. Preparar el entorno

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from PIL import Image
from IPython.display import display
import urllib.request

print("TensorFlow:", tf.__version__)

## 2. Descargar el conjunto de ejemplo desde GitHub

El notebook descarga directamente las **20 imágenes PNG**, organizadas en diez carpetas por categoría, desde el repositorio del curso. No necesitas descargar ZIP ni conectar Google Drive para esta primera parte.

Fuente: [diccionario de datos NB06_CIFAR10](https://github.com/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/02_Datasets/NB06_CIFAR10/README.md).

In [ ]:
BASE_RAW = "https://raw.githubusercontent.com/juliopez/Taller-Fundamentos-Data-Science-Python/main/Machine_Learning_2026/02_Datasets/NB06_CIFAR10"
CLASES_CIFAR10 = ["avion", "automovil", "pajaro", "gato", "ciervo", "perro", "rana", "caballo", "barco", "camion"]
DESTINO = Path("/content/NB06_CIFAR10")
registros = []
for clase in CLASES_CIFAR10:
    carpeta = DESTINO / clase
    carpeta.mkdir(parents=True, exist_ok=True)
    for numero in (1, 2):
        nombre = f"{clase}_{numero:02d}.png"
        ruta = carpeta / nombre
        if not ruta.exists():
            urllib.request.urlretrieve(f"{BASE_RAW}/{clase}/{nombre}", ruta)
        registros.append({"categoria_referencia": clase, "ruta": str(ruta), "imagen": nombre})
muestra = pd.DataFrame(registros)
assert len(muestra) == 20 and muestra["categoria_referencia"].nunique() == 10
print("Imágenes descargadas:", len(muestra))
display(muestra)

## 3. Observar el dataset antes de predecir

Revisa las imágenes y sus categorías originales. CIFAR-10 contiene imágenes de **32 × 32 píxeles**: al ampliarlas para MobileNetV2 no se recupera información visual perdida.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 5, figsize=(14, 10))
for ax, fila in zip(axes.flat, muestra.itertuples(index=False)):
    with Image.open(fila.ruta) as im:
        ax.imshow(im.convert("RGB").resize((128, 128)))
    ax.set_title(fila.categoria_referencia + "\n" + fila.imagen, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Conocer y cargar MobileNetV2 preentrenada

### ¿Qué es MobileNetV2?

**MobileNetV2** es una red neuronal convolucional presentada por investigadores de Google en **2018**. Fue diseñada para clasificar imágenes utilizando relativamente pocos recursos computacionales, por lo que resulta especialmente interesante para teléfonos móviles y dispositivos con capacidad limitada. Su arquitectura emplea bloques residuales invertidos y cuellos de botella lineales para procesar imágenes de forma eficiente.

### ¿Por qué utilizamos un modelo preentrenado?

Entrenar una CNN desde cero exige muchas imágenes etiquetadas, tiempo y capacidad de cómputo. En cambio, un **modelo preentrenado** ya aprendió patrones visuales a partir de un conjunto amplio de imágenes. Podemos reutilizarlo para hacer predicciones (**inferencia**) y analizar sus resultados sin volver a entrenarlo. Más adelante, técnicas como *Transfer Learning* permiten adaptar parte de ese conocimiento a categorías nuevas.

En esta práctica cargaremos MobileNetV2 con pesos entrenados en **ImageNet**, que permiten predecir entre **1.000 clases**. Esas clases no coinciden necesariamente con las diez categorías de CIFAR-10.

### ¿Cómo se comparan los modelos de clasificación?

Entre **2010 y 2017**, el desafío anual **ImageNet Large Scale Visual Recognition Challenge (ILSVRC)** impulsó grandes avances en clasificación de imágenes. Algunas arquitecturas históricas fueron **AlexNet (2012)**, **VGG (2014)** y **ResNet (2015)**; **MobileNetV2 (2018)** surgió después, con énfasis en la eficiencia.

Hoy no existe un único «ranking anual» universal: los modelos se comparan mediante distintos conjuntos de prueba y métricas, como **exactitud Top-1**, **exactitud Top-5**, cantidad de parámetros, latencia y consumo de recursos. Un modelo más preciso no siempre es el más conveniente para una aplicación concreta.

**En NB06 utilizaremos MobileNetV2 como ejemplo de modelo eficiente y accesible**, no porque sea necesariamente el mejor modelo disponible.

**Lecturas de referencia:** [artículo MobileNetV2 (2018)](https://arxiv.org/abs/1801.04381) · [historia de ILSVRC](https://www.image-net.org/challenges/LSVRC/) · [documentación de MobileNetV2 en Keras](https://keras.io/api/applications/mobilenet/).

La siguiente celda descarga los pesos preentrenados la primera vez que se ejecuta. **No entrenaremos ni modificaremos los pesos del modelo.**

In [ ]:
mobilenet = keras.applications.MobileNetV2(weights="imagenet", include_top=True)
print("Parámetros del modelo:", mobilenet.count_params())

## 5. Ejecutar las predicciones

Cada imagen se redimensiona a 224 × 224 y se preprocesa con la función específica de MobileNetV2. Mostraremos las **tres etiquetas ImageNet más probables** y la probabilidad de la primera. Estas probabilidades no equivalen a una garantía de acierto.

In [ ]:
def inferir_mobilenet(ruta):
    with Image.open(ruta) as im:
        x = np.asarray(im.convert("RGB").resize((224, 224)), dtype=np.float32)[None, ...]
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    probabilidades = mobilenet.predict(x, verbose=0)
    return keras.applications.mobilenet_v2.decode_predictions(probabilidades, top=3)[0]

def predecir_tabla(datos):
    filas = []
    for fila in datos.itertuples(index=False):
        preds = inferir_mobilenet(fila.ruta)
        filas.append({
            "categoria_referencia": fila.categoria_referencia,
            "imagen": fila.imagen,
            "prediccion_top1": preds[0][1],
            "probabilidad_top1": float(preds[0][2]),
            "top3": " | ".join(f"{etiqueta} ({prob:.1%})" for _, etiqueta, prob in preds),
        })
    return pd.DataFrame(filas)

resultados = predecir_tabla(muestra)
display(resultados)

## 6. Analizar aciertos y errores

Observa las imágenes y compara su categoría de referencia con la etiqueta ImageNet. **No uses una igualdad automática de nombres:** ImageNet puede emplear categorías más específicas o diferentes. Clasifica manualmente cada resultado como **coherente**, **parcialmente relacionado** o **incorrecto**, justificando tu criterio.

La siguiente tabla permite anotar tu evaluación sin modificar las predicciones.

In [ ]:
analisis = resultados[["categoria_referencia", "imagen", "prediccion_top1", "probabilidad_top1"]].copy()
analisis["evaluacion"] = ""  # coherente / parcialmente relacionado / incorrecto
analisis["justificacion"] = ""
display(analisis)
print("Completa las dos últimas columnas según tu análisis; no se calcula una exactitud automática.")

## 7. Actividad autónoma — dataset propio del NB05

Ahora utiliza **tu propio dataset del NB05** para repetir el experimento con imágenes diferentes a las de CIFAR-10. Conecta Google Drive y configura la carpeta que utilizaste en NB05, organizada en subcarpetas por categoría.

Se seleccionarán **hasta dos imágenes por categoría**, sin exigir un mínimo fijo de categorías o imágenes. Observa las predicciones y contrástalas con las etiquetas de tus carpetas.

Antes de ejecutar, cambia `DATASET_DIR` por la ruta real de tu dataset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = Path("/content/drive/MyDrive/MachineLearning2026/dataset")  # AJUSTAR a tu NB05
assert DATASET_DIR.is_dir(), f"No existe la carpeta: {DATASET_DIR}"
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
propios = []
for carpeta in sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()):
    imagenes = sorted(p for p in carpeta.rglob("*") if p.is_file() and p.suffix.lower() in EXT)
    for ruta in imagenes[:2]:
        propios.append({"categoria_referencia": carpeta.name, "ruta": str(ruta), "imagen": ruta.name})
assert propios, "No se encontraron imágenes compatibles en las carpetas de clases"
muestra_propia = pd.DataFrame(propios)
resultados_propios = predecir_tabla(muestra_propia)
print("Imágenes propias evaluadas:", len(muestra_propia))
display(resultados_propios)

analisis_propio = resultados_propios[["categoria_referencia", "imagen", "prediccion_top1", "probabilidad_top1"]].copy()
analisis_propio["evaluacion"] = ""  # coherente / parcialmente relacionado / incorrecto
analisis_propio["justificacion"] = ""
display(analisis_propio)

## 8. Subir una imagen individual desde tu computador

Esta actividad permite probar MobileNetV2 con **una imagen nueva**, sin agregarla al dataset ni volver a entrenar el modelo.

1. Ejecuta la siguiente celda y selecciona una imagen JPG, PNG, BMP o WEBP.
2. Observa la imagen y sus **tres predicciones ImageNet más probables**.
3. Explica en una celda Markdown si la clasificación tiene sentido y por qué una probabilidad alta no garantiza un acierto.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()
assert archivos_subidos, "No se seleccionó ninguna imagen"
nombre_archivo = next(iter(archivos_subidos))
ruta_individual = Path("/content") / nombre_archivo
with Image.open(ruta_individual) as im:
    imagen_individual = im.convert("RGB")
    display(imagen_individual.copy().resize((224, 224)))

predicciones_individuales = inferir_mobilenet(ruta_individual)
tabla_individual = pd.DataFrame([
    {"posicion": i, "etiqueta_imagenet": etiqueta, "probabilidad": float(prob)}
    for i, (_, etiqueta, prob) in enumerate(predicciones_individuales, start=1)
])
print("Imagen analizada:", nombre_archivo)
display(tabla_individual)

## 9. Preguntas para responder en celdas Markdown

**A.** Identifica dos imágenes de CIFAR-10: ¿qué predijo MobileNetV2 y con qué probabilidad? ¿Son coherentes las predicciones?

**B.** Describe un caso en que la etiqueta ImageNet sea más específica o distinta de la categoría CIFAR-10.

**C.** ¿Cómo influye la resolución original de 32 × 32 píxeles? ¿Por qué redimensionar no recupera información perdida?

**D.** **Dataset propio del NB05 (obligatorio):** identifica dos predicciones, evalúa su coherencia con las categorías reales y compara con el conjunto CIFAR-10.

**E.** **Imagen individual:** indica qué imagen subiste, cuáles fueron las predicciones y si el resultado tiene sentido.

**F.** ¿Por qué una probabilidad alta no garantiza un acierto? ¿Cómo podría ayudar el *Transfer Learning* a adaptar un modelo a tus categorías, sin implementarlo?

## 10. Cierre y continuidad

En este NB06 has evaluado **MobileNetV2 preentrenada** con un conjunto común, **tu dataset del NB05** y una **imagen individual subida a Colab**, sin reentrenar pesos.

**NB07:** preparación e interoperabilidad de un modelo común de piedra, papel o tijera. **NB08:** despliegue del modelo con Streamlit.

## 11. Para seguir aprendiendo: ¿podemos adaptar MobileNetV2 a nuestras imágenes?

**Sí.** Hasta ahora utilizamos MobileNetV2 como un modelo que ya sabe reconocer muchos objetos. Le mostramos imágenes nuevas y observamos sus respuestas, pero **no le enseñamos categorías nuevas**.

Piensa en alguien que ya aprendió a reconocer cientos de objetos. Si ahora queremos que distinga tres gestos específicos —**piedra, papel y tijera**—, no necesita aprender desde cero qué son las formas, los bordes o las texturas. Puede aprovechar ese conocimiento y aprender a distinguir los tres gestos.

Eso es lo que buscamos al adaptar un **modelo preentrenado**.

### Tres ideas que no debemos confundir

- **Inferencia (lo que hicimos en NB06):** mostramos una imagen y el modelo entrega una predicción. **Sus pesos no cambian**; por eso no aprende de las imágenes que probamos.
- **Transfer Learning (aprendizaje por transferencia):** aprovechamos las características que MobileNetV2 ya aprendió y añadimos una nueva capa de clasificación para nuestras categorías. Primero entrenamos **solo esa capa nueva**; las capas originales permanecen congeladas.
- **Fine-Tuning (ajuste fino):** después de entrenar la nueva capa, podemos **descongelar algunas capas finales** de MobileNetV2 y continuar el entrenamiento con cuidado. Así modificamos una parte de sus pesos para que se adapte mejor a nuestras imágenes.

**Importante:** reemplazar la capa de clasificación y entrenarla no es todavía, por sí solo, el ajuste fino de las capas originales. Es la primera etapa del aprendizaje por transferencia. El *Fine-Tuning* comienza cuando permitimos que algunas capas preentrenadas también actualicen sus pesos.

### Ejemplo: reconocer piedra, papel y tijera

MobileNetV2 fue entrenada para clasificar **1.000 categorías de ImageNet**, no específicamente nuestras tres categorías. Si queremos que sus respuestas sean «piedra», «papel» o «tijera», debemos preparar imágenes etiquetadas y adaptar la salida del modelo.

### ¿Qué pasos seguiríamos para hacerlo?

1. **Preparar las imágenes.** Reunir ejemplos de piedra, papel y tijera, cada uno en su carpeta. Separar las imágenes en conjuntos de **entrenamiento, validación y prueba**. La prueba se reserva para comprobar el resultado final con imágenes que el modelo no haya utilizado para aprender.
2. **Cargar MobileNetV2 preentrenada.** Aprovechar sus capas para reconocer características visuales, pero cargarla **sin su capa original de clasificación de 1.000 categorías**.
3. **Agregar una nueva salida.** Incorporar una capa capaz de entregar **tres resultados**: piedra, papel y tijera.
4. **Entrenar primero la nueva capa.** Mantener congeladas las capas originales de MobileNetV2. En esta etapa, solo se actualizan los pesos de las capas nuevas. Este paso se conoce como **extracción de características** dentro del aprendizaje por transferencia.
5. **Realizar Fine-Tuning.** Si es necesario mejorar la adaptación, descongelar **algunas de las últimas capas** de MobileNetV2 y continuar el entrenamiento con una **tasa de aprendizaje pequeña**, para evitar alterar bruscamente lo que el modelo ya aprendió.
6. **Evaluar y comparar.** Medir el rendimiento con las imágenes de prueba y compararlo con el resultado anterior al ajuste fino. Podemos utilizar, por ejemplo, **exactitud (accuracy)** y una **matriz de confusión** para identificar qué gestos se confunden.

### ¿Siempre mejora el modelo al realizar Fine-Tuning?

**No necesariamente.** Si tenemos pocas imágenes, categorías mal etiquetadas o entrenamos demasiado, el modelo puede memorizar ejemplos y funcionar peor con imágenes nuevas. A esto se le llama **sobreajuste**. Por eso es importante comparar los resultados antes y después, en lugar de asumir que más entrenamiento siempre significa mejores predicciones.

### Relación con los notebooks del curso

En **NB05** construimos y entrenamos una CNN propia. En **NB06** utilizamos una CNN preentrenada para hacer predicciones **sin modificar sus pesos**. El aprendizaje por transferencia y el *Fine-Tuning* serían un **experimento posterior**: adaptar MobileNetV2 y comparar su desempeño con nuestra CNN.

**Esta sección es una propuesta para profundizar, no una actividad que debas ejecutar ahora.** En **NB07** continuaremos con la preparación e interoperabilidad de un modelo ya entrenado; en **NB08**, con su despliegue en Streamlit.


### Laboratorio opcional: probar los pasos 3 a 6 con código

Si tienes tiempo y quieres experimentar, puedes ejecutar las siguientes celdas **en el mismo Colab**, después de completar la sección 7. Usaremos **tu dataset del NB05**, cuya ruta ya configuraste en `DATASET_DIR`. No necesitas disponer de imágenes de piedra, papel y tijera: el procedimiento funciona con las categorías de tus carpetas.

**Antes de comenzar:** cada carpeta debe contener varias imágenes; idealmente, al menos cinco por categoría para este ejercicio. El código separa imágenes para **entrenamiento** y **validación**. La validación sirve para observar el aprendizaje, pero **no reemplaza un conjunto de prueba independiente** si queremos evaluar rigurosamente el modelo final.

#### Paso 3. Preparar los datos y agregar una nueva capa de clasificación

La primera celda lee tus carpetas y crea dos conjuntos de imágenes. Luego carga MobileNetV2 **sin su clasificador original** y agrega una salida con tantas categorías como carpetas tenga tu dataset. Las capas originales quedan congeladas: todavía no estamos haciendo Fine-Tuning.

In [ ]:
# Reutiliza DATASET_DIR, configurada en la sección 7.
# Si ejecutas este laboratorio de manera independiente, define antes esa ruta.
assert "DATASET_DIR" in globals() and DATASET_DIR.is_dir(), (
    "Primero ejecuta la sección 7 y configura DATASET_DIR con tu dataset del NB05."
)

TAMANO_IMAGEN = (160, 160)
LOTE = 16
SEMILLA = 42

entrenamiento = tf.keras.utils.image_dataset_from_directory(
    str(DATASET_DIR), validation_split=0.2, subset="training",
    seed=SEMILLA, image_size=TAMANO_IMAGEN, batch_size=LOTE
)
validacion = tf.keras.utils.image_dataset_from_directory(
    str(DATASET_DIR), validation_split=0.2, subset="validation",
    seed=SEMILLA, image_size=TAMANO_IMAGEN, batch_size=LOTE
)
categorias = entrenamiento.class_names
assert len(categorias) >= 2, "Se necesitan al menos dos categorías."
assert len(validacion.file_paths) > 0, "Se necesitan más imágenes para validar."
print("Categorías:", categorias)

# Modelo nuevo e independiente del MobileNetV2 usado antes en NB06.
base = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3), include_top=False, weights="imagenet"
)
base.trainable = False

entrada = tf.keras.Input(shape=(160, 160, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(entrada)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
salida = tf.keras.layers.Dense(len(categorias), activation="softmax")(x)
modelo_adaptado = tf.keras.Model(entrada, salida)
print("Nueva salida:", len(categorias), "categorías")

#### Paso 4. Entrenar únicamente la nueva capa

En este primer entrenamiento, MobileNetV2 conserva sus conocimientos originales. **Solo aprende la capa nueva** que añadimos para distinguir nuestras categorías. Observa la exactitud de entrenamiento y validación al finalizar.

In [ ]:
modelo_adaptado.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
historial_transferencia = modelo_adaptado.fit(
    entrenamiento, validation_data=validacion, epochs=3
)
print("Exactitud de validación al terminar la primera etapa:",
      round(historial_transferencia.history["val_accuracy"][-1], 3))

#### Paso 5. Realizar Fine-Tuning de algunas capas finales

Ahora sí permitimos que **una parte pequeña de MobileNetV2 modifique sus pesos**. Mantenemos congeladas las capas anteriores y usamos una tasa de aprendizaje mucho menor para evitar cambios bruscos. Las capas de normalización por lotes (*BatchNormalization*) se mantienen congeladas, una precaución importante en este tipo de ajuste.

Este paso puede mejorar o empeorar el resultado: **no está garantizado que aumente la exactitud**.

In [ ]:
base.trainable = True
for capa in base.layers[:-20]:
    capa.trainable = False
for capa in base.layers[-20:]:
    if isinstance(capa, tf.keras.layers.BatchNormalization):
        capa.trainable = False

# Después de cambiar qué capas son entrenables, debemos recompilar.
modelo_adaptado.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
historial_ajuste = modelo_adaptado.fit(
    entrenamiento, validation_data=validacion, epochs=3
)
print("Exactitud de validación después del Fine-Tuning:",
      round(historial_ajuste.history["val_accuracy"][-1], 3))

#### Paso 6. Comparar los resultados

La siguiente celda muestra la exactitud de validación antes y después del Fine-Tuning. **Una diferencia pequeña no demuestra por sí sola que un método sea mejor**, especialmente con pocos datos. Para una evaluación final confiable, habría que reservar imágenes de **prueba independientes**, no utilizadas para entrenar ni para tomar decisiones durante el ajuste.

In [ ]:
exactitud_antes = historial_transferencia.history["val_accuracy"][-1]
exactitud_despues = historial_ajuste.history["val_accuracy"][-1]
comparacion = pd.DataFrame({
    "Etapa": ["Transfer Learning (capa nueva)", "Fine-Tuning (capas finales)"],
    "Exactitud de validación": [exactitud_antes, exactitud_despues]
})
display(comparacion)
print("Diferencia:", f"{exactitud_despues - exactitud_antes:+.2%}")
print("Recuerda: la validación no sustituye una evaluación con datos de prueba independientes.")

**Para reflexionar:** ¿mejoró la exactitud de validación después del Fine-Tuning? ¿Crees que ese cambio sería igual con otras imágenes? ¿Qué sucedería si tu dataset fuera muy pequeño?

**Este laboratorio es completamente opcional.** No cambia los objetivos ni los resultados de las actividades anteriores del NB06. En NB07 seguiremos con interoperabilidad y ONNX, no con entrenamiento.